In [ ]:
import shutil
import tempfile
from pathlib import Path

from matplotlib import pyplot as plt

from pycohortflow import (
    export,
    gradient_palette,
    plot_and_export,
    plot_cfd,
)

## Output directory

All saved figures from this notebook are written to a temporary
directory.  The final cell removes the directory so the example
leaves no files on disk.

In [ ]:
tmp_dir = Path(tempfile.mkdtemp(prefix="pycohortflow_example_", dir=Path.cwd()))
print(f"Saving figures to: {tmp_dir}")

## Cohort data

In [ ]:
cohort_data = [
    {
        "heading": "Registered Patients",
        "description": "Total patients registered in database",
        "N": 350,
    },
    {
        "heading": "Screening",
        "description": "Underwent initial screening",
        "N": 150,
        "exclusion_description": "Did not meet inclusion criteria",
    },
    {
        "heading": "Eligible",
        "description": "Medically cleared for procedure",
        "N": 120,
        "exclusion_description": "Declined to participate / Lost to follow-up",
    },
    {
        "heading": "Final Analysis",
        "N": 115,
        "exclusion_description": "Data incomplete",
    },
]

## White style

The default look — clean white boxes, bold headings, side card
for each exclusion.

In [ ]:
fig, ax = plot_cfd(
    cohort_data,
    save_dir=tmp_dir,
    img_name="clinical_flow_chart_white",
    save_format=["png", "pdf", "svg"],
    figure_title="Clinical Cohort Flow Diagram (white)",
    figsize=(10, 8),
    dpi=200,
    style="white",
)
plt.show()
plt.close(fig)

## Colorful style

Pastel gradient backgrounds for the main and exclusion boxes.

In [ ]:
fig, ax = plot_cfd(
    cohort_data,
    save_dir=tmp_dir,
    img_name="clinical_flow_chart_colorful",
    save_format=["png", "pdf", "svg"],
    figure_title="Clinical Cohort Flow Diagram (colorful)",
    figsize=(10, 8),
    dpi=200,
    style="colorful",
)
plt.show()
plt.close(fig)

## Minimal Style (Colorful)

Notebook-style look (white boxes, normal-weight headings, italic
side text instead of an exclusion box) — but with every node
filled by a pastel gradient that mirrors the ``colorful`` style.
The first and last boxes are bolded to highlight the start and
end of the cohort.

In [ ]:
colorful_data = [dict(d) for d in cohort_data]
node_palette = gradient_palette("#dff1ff", "#dff7e8", len(colorful_data))
for node, color in zip(colorful_data, node_palette):
    node["color"] = color
colorful_data[0]["heading_fontweight"] = "bold"
colorful_data[-1]["heading_fontweight"] = "bold"

fig, ax = plot_cfd(
    colorful_data,
    save_dir=tmp_dir,
    img_name="clinical_flow_chart_minimal_colorful",
    save_format=["png", "pdf", "svg"],
    figure_title="Clinical Cohort Flow Diagram (minimal · colorful)",
    figsize=(10, 8),
    dpi=200,
    style="minimal",
)
plt.show()
plt.close(fig)

## Minimal Style (White)

In [ ]:
white_minimal_data = [dict(d) for d in cohort_data]
for node in white_minimal_data:
    node["color"] = "white"
white_minimal_data[0]["heading_fontweight"] = "bold"
white_minimal_data[-1]["heading_fontweight"] = "bold"

fig, ax = plot_cfd(
    white_minimal_data,
    save_dir=tmp_dir,
    img_name="clinical_flow_chart_minimal_white",
    save_format=["png", "pdf", "svg"],
    figure_title="Clinical Cohort Flow Diagram (minimal · white)",
    figsize=(10, 8),
    dpi=200,
    style="minimal",
)
plt.show()
plt.close(fig)

## Cleanup

Remove the temporary directory and all files it contains.

## Export for the Interactive Generator

Render a diagram **and** write a `.cohort.json` + `.style.toml`
pair that can be pasted into the browser-based
[Interactive Generator](https://fschwar4.github.io/pycohortflow/generator.html)
to reproduce the same figure without Python.

Two ways to do it:

1. **`plot_and_export(...)`** — convenience wrapper around
   `plot_cfd` and `export`.  Renders, saves the figure, and
   writes the export pair in one call.
2. **`export(...)`** — standalone.  Use when you already have a
   `plot_cfd` call and want to add the export side without
   refactoring.

### Combined plot + export

In [ ]:
fig, ax, exp_result = plot_and_export(
    cohort_data,
    out_dir=tmp_dir,
    name="clinical_flow_chart_export_demo",
    style="colorful",
    save_format=["png", "pdf", "svg"],
    figure_title="Clinical Cohort Flow Diagram (export demo)",
    figsize=(10, 8),
    dpi=200,
)
plt.show()
plt.close(fig)

print(f"Figure  : {exp_result['json_path'].with_suffix('.png').name}")
print(f"Cohort  : {exp_result['json_path'].name}")
print(f"Style   : {exp_result['toml_path'].name}")
print(f"Paste   : open the Interactive Generator, paste the .cohort.json into")
print(f"          the data textarea and the .style.toml into TOML overrides.")

### Standalone export (no figure)

Use when the figure is already rendered and you only need the
JSON/TOML pair.

In [ ]:
result = export(
    cohort_data,
    style="minimal",
    figure_title="Clinical Cohort Flow Diagram (minimal)",
    out_dir=tmp_dir,
    basename="clinical_flow_chart_export_minimal",
)

print("--- _meta block of cohort.json ---")
print("\n".join(result["cohort_json"].splitlines()[:8]))
print("...")
print()
print("--- first lines of style.toml ---")
print("\n".join(result["style_toml"].splitlines()[:6]))

In [ ]:
shutil.rmtree(tmp_dir)
print(f"Removed temporary directory: {tmp_dir}")